# 02 — ReAct agent

**Definition:** interleave thinking and doing, discovering the path while executing.
ReAct = **Rea**son + **Act** (Yao et al., 2022).

This is the one I reach for by default. It solves most real problems, and 03–07 are all
"ReAct plus a bit more".

```
        +---------------------------+
        |                           |
        v                           |
     Reason  --->  Act  --->  Observe
   (LLM picks    (tool runs)  (result goes
    an action)                 back to LLM)
        |
        | no more tool calls needed
        v
      Answer
```

The loop runs **until the LLM stops asking for tools**. Nobody knows in advance how many
iterations that takes — that is the entire point, and also the entire risk.

The graph I'm building is only two nodes:

```
        START
          |
          v
      +-------+
   +->| agent |   (LLM: reason + choose)
   |  +-------+
   |      | conditional edge
   |      +--- has tool_calls? --> +-------+
   |      |                        | tools |
   |      |                        +-------+
   |      |                            |
   +------+----------------------------+     <- THE CYCLE
          |
          +--- no tool_calls ---------> END
```

## Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


def show(graph):
    print(graph.get_graph().draw_mermaid())


print(llm.invoke("Reply with the single word: ready").content)

## Tools

Tool design matters more than the prompt. What I keep reminding myself:

1. The **docstring is the spec the LLM reads**. Write it for the model, not for a human.
2. **Type hints become the JSON schema.** Untyped args = broken tool.
3. Keep each tool narrow. One tool that does five things confuses the model.
4. Return strings or simple values — they get serialised into the message list.
5. Handle errors *inside* the tool and return the error text, so the LLM can self-correct.

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use for any weather-related question."""
    data = {
        "cologne": "12C, light rain",
        "vienna": "18C, sunny",
        "salzburg": "9C, overcast",
    }
    # Return a friendly miss instead of raising -> the LLM can recover on its own.
    return data.get(city.lower(), f"No weather data for {city}")


@tool
def get_activities(city: str, weather_condition: str) -> str:
    """Suggest activities for a city given the weather condition
    (e.g. 'rain', 'sunny', 'snow'). Call get_weather FIRST to learn the condition."""
    if "rain" in weather_condition.lower():
        return f"Indoor picks in {city}: museums, thermal baths, cafes"
    return f"Outdoor picks in {city}: parks, walking tours, river promenade"


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '23 * 4 + 10'."""
    try:
        # eval is fine for a demo; use asteval/numexpr for anything real.
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"      # the LLM sees this and retries with a fixed expression


tools = [get_weather, get_activities, calculator]

# bind_tools() attaches the JSON schemas so the model CAN emit tool_calls.
# It does not execute anything - executing is my job (or ToolNode's).
llm_with_tools = llm.bind_tools(tools)
print("bound:", [t.name for t in tools])

### What does a tool call actually look like?

This is the thing that made the pattern click for me: the LLM doesn't *run* anything.
It just *asks*, and something else has to answer.

In [ ]:
resp = llm_with_tools.invoke("What's the weather in Cologne?")
print("content   :", repr(resp.content))     # usually empty when a tool is requested
print("tool_calls:", resp.tool_calls)        # <- the "Action" from the ReAct paper

And the generated schema the model actually sees — this is why docstrings and type hints matter:

In [ ]:
import json

print(json.dumps(get_activities.args_schema.model_json_schema(), indent=2))

## State

`MessagesState` is a prebuilt shortcut for:

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

The `add_messages` reducer is what makes ReAct possible at all: without it, each node's
returned list **replaces** the history instead of appending, so the LLM never sees the
tool result it just asked for. It also de-duplicates by message `id`, which is how you
overwrite or delete a message later (see notebook 05).

In [ ]:
from langgraph.graph import MessagesState


class ReActState(MessagesState):
    """MessagesState already gives me `messages` with the add_messages reducer.
    Subclass it to add extra keys - here a counter so I can see the loop length."""

    step_count: int

## Node A — `agent` (this IS the "Reason" step)

There is no separate "Thought" node. In the paper ReAct was a *prompt format*
(`Thought: ... / Action: ... / Observation: ...`); in LangGraph the thought is implicit in
the LLM's decision to emit a `tool_call` or not.

- **Thought** = the LLM call
- **Action** = the `tool_call` it emits
- **Observation** = the `ToolMessage` that comes back

Two nodes are enough because thinking and choosing are the same call.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM = SystemMessage(
    content=(
        "You are a helpful travel assistant. Use tools when you need facts. "
        "If one tool's output is needed as another tool's input, call them in sequence."
    )
)


def agent_node(state: ReActState) -> dict:
    """Reason over the FULL history and either emit tool_calls (loop continues)
    or plain text (loop ends)."""
    # The accumulated message list IS the agent's working memory.
    response = llm_with_tools.invoke([SYSTEM] + state["messages"])
    return {"messages": [response], "step_count": state.get("step_count", 0) + 1}

## Node B — `tools` (the "Act" + "Observe" step)

`ToolNode` does four things I'd otherwise hand-roll:

1. reads `tool_calls` off the last `AIMessage`
2. runs all of them (in parallel if there are several)
3. wraps each result in a `ToolMessage` with the matching `tool_call_id`
4. catches exceptions and returns them as content so the LLM can recover

That `tool_call_id` bookkeeping is not optional — every `tool_call` **must** be answered by
a `ToolMessage` with the same id or the provider returns a 400.

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

# The manual equivalent, so the mechanism is visible:
from langchain_core.messages import ToolMessage

BY_NAME = {t.name: t for t in tools}


def tool_node_manual(state):
    out = []
    for call in state["messages"][-1].tool_calls:
        result = BY_NAME[call["name"]].invoke(call["args"])
        out.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return {"messages": out}       # the id MUST match, or the API errors

Side quest: I tried calling `tool_node.invoke({"messages": [...]})` directly and got

```
ValueError: Missing required config key 'N/A' for 'tools'.
```

`ToolNode` expects a graph runtime around it (that's how `store` / `config` get injected),
so to exercise it standalone I have to wrap it in a one-node graph.

In [ ]:
from langgraph.graph import END, START, StateGraph

_tb = StateGraph(MessagesState)
_tb.add_node("tools", tool_node)
_tb.add_edge(START, "tools")
_tb.add_edge("tools", END)
_probe_graph = _tb.compile()

ai_msg = llm_with_tools.invoke("Weather in Vienna?")
print("ToolNode:", _probe_graph.invoke({"messages": [ai_msg]})["messages"][-1].content)
print("manual  :", tool_node_manual({"messages": [ai_msg]})["messages"][0].content)
print("\nboth produce a ToolMessage carrying tool_call_id:", ai_msg.tool_calls[0]["id"])

## The router

One function, and it's the heart of the pattern. Termination is purely a shape check on
the last message:

- `AIMessage` **with** `.tool_calls` -> go to `tools` ("I need more info")
- `AIMessage` **without** -> `END` ("I'm done, here's the answer")

No counter, no plan, no goal check. If the LLM never stops, `recursion_limit` is the only
thing standing between me and a very expensive afternoon.

In [ ]:
from langgraph.graph import END


def should_continue(state: ReActState) -> str:
    last = state["messages"][-1]
    # getattr guard: only AIMessage has .tool_calls
    if getattr(last, "tool_calls", None):
        return "tools"
    return END


# LangGraph ships this exact logic as `tools_condition`:
#   from langgraph.prebuilt import tools_condition
# I wrote it by hand once so I know what it does.

## Build the graph — the back-edge is the whole pattern

In [ ]:
from langgraph.graph import START, StateGraph

builder = StateGraph(ReActState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])
builder.add_edge("tools", "agent")     # <- this ONE line turns a chain into ReAct

react_graph = builder.compile()
show(react_graph)

## Run it — watch it discover a dependency

This query forces multi-step reasoning: it can't suggest activities before it knows the
weather. Nobody told it that ordering; it works it out from the tool docstrings.

In [ ]:
result = react_graph.invoke(
    {
        "messages": [HumanMessage(content="What should I do in Cologne today, given the weather?")],
        "step_count": 0,
    }
)

for m in result["messages"]:
    m.pretty_print()

print(f"\nLLM calls made: {result['step_count']}")

### Same loop, streamed

Two *independent* questions this time (arithmetic + weather), so nothing forces an order.
I expected one batched `AIMessage` with two `tool_calls`; what I actually get is two
separate agent -> tools laps. Whether a model batches parallel calls or serialises them is
a model behaviour, not a LangGraph one — so never write a router that assumes either.

In [ ]:
for chunk in react_graph.stream(
    {
        "messages": [HumanMessage(content="What is 23*4+10, and what's the weather in Vienna?")],
        "step_count": 0,
    },
    stream_mode="updates",
):
    for node, update in chunk.items():
        msg = update["messages"][-1]
        calls = getattr(msg, "tool_calls", None)
        print(f"[{node}] {calls if calls else msg.content[:150]}")

## The prebuilt version

Everything above in three lines. This is what I'd actually use at work; hand-rolling is
for when I need custom nodes inside the loop.

**Naming, because the tutorials disagree with each other:** this used to be
`langgraph.prebuilt.create_react_agent`. In LangGraph v1 it moved and was renamed to
`langchain.agents.create_agent` — the old import still works but warns
`LangGraphDeprecatedSinceV10`. The kwarg is `system_prompt` now (it was `prompt`, and
before that `state_modifier`).

In [ ]:
from langchain.agents import create_agent

quick_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful travel assistant.",
)

out = quick_agent.invoke({"messages": [{"role": "user", "content": "Weather in Salzburg?"}]})
print(out["messages"][-1].content)

Kwargs worth remembering:

| kwarg | does |
|---|---|
| `system_prompt` | system prompt (str or `SystemMessage`) |
| `checkpointer` | cross-turn memory (notebook 05) |
| `store` | long-term memory across threads (notebook 05) |
| `response_format` | force a structured final answer |
| `middleware` | hooks around the model call — trimming, guardrails, retries |
| `interrupt_before` | human-in-the-loop gate before a node (notebook 07) |

## Safety — the recursion limit

Not optional. A ReAct agent with a flaky tool loops forever, and each loop is billable.
Setting it deliberately low here so I can see the guard fire.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    react_graph.invoke(
        {"messages": [HumanMessage(content="What should I do in Cologne today?")], "step_count": 0},
        config={"recursion_limit": 2},
    )
except GraphRecursionError as e:
    print("GraphRecursionError ->", e)

Rule of thumb: `recursion_limit ~= 2 * (max expected tool calls) + 1`, because each loop
burns two steps (agent + tools).

## Notes to self

**How it ends:** an `AIMessage` with no `tool_calls`. That's it. There is no goal check —
notebook 07 adds one.

**ReAct is not "tool calling".** Tool calling is the mechanism; ReAct is the loop around it.
Reactive + tools = call once, format, stop (notebook 01). ReAct = feed the result back and
let the LLM decide again. The back-edge is the entire difference.

**Failure modes I've hit:**

| symptom | cause | fix |
|---|---|---|
| history vanishes each turn | no `add_messages` reducer | use `MessagesState` |
| provider 400 on tool use | a `tool_call` with no matching `ToolMessage` | use `ToolNode` |
| loops forever | flaky tool, LLM keeps retrying | `recursion_limit`, and return errors as text |
| picks the wrong tool | vague docstrings, too many tools | rewrite docstrings; split into agents (06) |

**API I used:**

```python
@tool                                   # typed args + docstring = the schema
llm.bind_tools(tools)                   # attach schemas (does not execute)
ToolNode(tools)                         # execute + wrap in ToolMessage
class S(MessagesState): ...             # messages with add_messages
tools_condition                         # prebuilt router -> "tools" | END
builder.add_edge("tools", "agent")      # the cycle
create_agent(model=llm, tools=tools, system_prompt=...)   # langchain.agents
config={"recursion_limit": N}
```

Next: **03 — Planning**, where the agent writes the whole plan up front instead of
discovering it one step at a time.